# WMT25 MIST LR — Text Diversity Analyses (5 & 6)

Analyses 5 and 6 both use Shaib et al. (2024) *Standardizing the Measurement of Text Diversity* (`diversity` package). They are kept together in this dedicated notebook to keep `analysis_lr.ipynb` manageable.

- **Analysis 5** — Intra-response repetitiveness: how much does a model repeat itself *within* a single response?
- **Analysis 6** — Between-response diversity: how much does a model vary its writing *between* responses?

This notebook is self-contained: it rebuilds `df` from scratch (≈30 s) before running either analysis.

---
## 0. Setup — rebuild `df`

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sacrebleu.metrics import CHRF

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})

# Reproducibility for any stochastic operations later
import random
random.seed(42)
np.random.seed(42)

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT        = Path("/Users/ritaberrada/Desktop/wmt-mist")
HUMEVAL_LR  = ROOT / "data/humeval/lr.json"
AGG_LR      = ROOT / "data/humeval_aggregated/lr.json"
AGG_OEG     = ROOT / "data/humeval_aggregated/oeg.json"
AGG_XLSUM   = ROOT / "data/humeval_aggregated/xlsum.json"
SUBS_DIR    = ROOT / "data/submissions"

# ── Top 8 models (top-5 + both models tied at rank 6) ─────────────────────────
# File stem → display name
TOP_MODELS = {
    "Gemini-2.5-Pro" : "Gemini 2.5 Pro",
    "Claude-4"       : "Claude 4",
    "DeepSeek-V3"    : "DeepSeek V3",
    "GPT-4.1"        : "GPT 4.1",
    "Llama-4-Maverick": "Llama 4 Maverick",
    "CommandA"       : "CommandA",
    "Mistral-Medium" : "Mistral Medium",
}

STEMS   = list(TOP_MODELS.keys())      # short keys used in column names
DISPLAY = list(TOP_MODELS.values())    # human-readable model names

print("Models to analyse:")
for k, v in TOP_MODELS.items():
    print(f"  {v:<22}  ({SUBS_DIR / (k + '.json')}).name")

In [ ]:
with open(HUMEVAL_LR) as f:
    lr_raw = json.load(f)

df_lr = pd.DataFrame(lr_raw)
df_lr.index.name = "N"          # N = position in the list = suffix in taskid
df_lr = df_lr.reset_index()     # N becomes a regular column

print(f"Total LR items: {len(df_lr)}")
print(f"Columns: {list(df_lr.columns)}")
df_lr.head(3)

In [ ]:
# ── Load aggregated scores + filter to our 7 models ────────────────────────
with open(AGG_LR) as f:
    agg_raw = json.load(f)

df_agg = pd.DataFrame(agg_raw).T.astype(float)
df_agg.index.name = 'language'
avg_row = df_agg.loc['Average'].sort_values(ascending=False)
df_agg = df_agg[avg_row.index]

df_agg_top = df_agg[DISPLAY].copy()
print(f'df_agg_top shape: {df_agg_top.shape}')
df_agg_top.round(1)

In [ ]:
# ── Submission loader + scoring utilities ───────────────────────────────────
def load_lr_entries(stem: str) -> dict:
    """
    Load LR entries for one model. Returns a dict: N -> entry dict.
    
    token handling:
      - Most models: tokens = {input_tokens, output_tokens, thinking_tokens, finish_reason}
      - TowerPlus / EuroLLM: tokens = bare int (not in our top 8, but handle gracefully)
      - Some Gemini entries: tokens = None
    """
    path = SUBS_DIR / f"{stem}.json"
    with open(path) as f:
        data = json.load(f)
    
    lr_entries = {}
    for entry in data:
        tid = entry["taskid"]
        if not tid.startswith("linguistic_reasoning_"):
            continue
        # N is always the integer after the last underscore
        N = int(tid.rsplit("_", 1)[-1])
        lr_entries[N] = entry
    
    return lr_entries


# Load all top-8 models
submissions = {stem: load_lr_entries(stem) for stem in STEMS}

for stem, entries in submissions.items():
    print(f"  {TOP_MODELS[stem]:<22} → {len(entries)} LR entries")

_chrf = CHRF()


def extract_answer(text: str) -> str:
    """
    Extract the final bracketed answer from a model response.
    
    Strategy:
      1. Find all [...] occurrences (non-nested regex).
      2. Return the LAST match — handles multi-bracket reasoning traces.
      3. Fallback: last non-empty line (catches models that ignore bracket format).
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    matches = re.findall(r'\[([^\[\]]*)\]', text)
    if matches:
        return matches[-1].strip()
    # Fallback: last non-empty line
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    return lines[-1] if lines else ""


def score_item(pred: str, gold: str, eval_type: str, points: float) -> float:
    """
    Score one item.
    
    exact match → 1.0 if pred.lower() == gold.lower(), else 0.0  (× points)
    chrF        → sacrebleu sentence-level chrF / 100            (× points)
    """
    pred = pred.strip()
    gold = gold.strip()
    if eval_type == "chrF":
        raw = _chrf.sentence_score(pred, [gold]).score / 100.0
    else:  # exact match
        raw = 1.0 if pred.lower() == gold.lower() else 0.0
    return raw * points


print("Utilities defined. Quick smoke-test:")
print(f"  extract_answer('Let me think... [Paris]') → '{extract_answer('Let me think... [Paris]')}'")
print(f"  extract_answer('The answer is Paris.')    → '{extract_answer('The answer is Paris.')}'")
print(f"  score_item('paris','Paris','exact match',1.0) → {score_item('paris','Paris','exact match',1.0)}")
print(f"  score_item('',    'Paris','exact match',1.0) → {score_item('',    'Paris','exact match',1.0)}")
print(f"  score_item('Paris','Paris','chrF',2.0)       → {score_item('Paris','Paris','chrF',2.0):.3f}")

In [ ]:
# Start from the gold metadata
df = df_lr[["N", "id", "points", "instruction_language",
            "problem_language", "type", "eval_type", "answer"]].copy()

for stem in STEMS:
    entries = submissions[stem]
    scores, tok_out, tok_think, brackets, preds, raws = [], [], [], [], [], []

    for _, row in df_lr.iterrows():
        N         = row["N"]
        gold      = row["answer"]
        eval_type = row["eval_type"]
        points    = row["points"]

        entry = entries.get(N)
        if entry is None:
            # Missing entry (should not happen for top-8)
            scores.append(0.0)
            tok_out.append(np.nan)
            tok_think.append(0)
            brackets.append(False)
            preds.append("")
            raws.append("")
            continue

        raw    = entry.get("answer") or ""
        tokens = entry.get("tokens")

        # Token extraction — guard against None and bare-int formats
        if isinstance(tokens, dict):
            ot = tokens.get("output_tokens", np.nan)
            tt = tokens.get("thinking_tokens", 0) or 0
        else:
            ot = np.nan   # bare int or None → not comparable across models
            tt = 0

        pred = extract_answer(raw)
        sc   = score_item(pred, gold, eval_type, points)
        has_bracket = bool(re.search(r'\[([^\[\]]*)\]', raw))

        scores.append(sc)
        tok_out.append(ot)
        tok_think.append(tt)
        brackets.append(has_bracket)
        preds.append(pred)
        raws.append(raw)

    df[f"score_{stem}"]        = scores
    df[f"tokens_out_{stem}"]   = tok_out
    df[f"tokens_think_{stem}"] = tok_think
    df[f"bracket_{stem}"]      = brackets
    df[f"pred_{stem}"]         = preds
    df[f"raw_{stem}"]          = raws

    print(f"  ✓ {TOP_MODELS[stem]}")

print(f"\ndf shape: {df.shape}")
df.head(3)

In [ ]:
# ── Analysis 1 Step 0: Setup ───────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'lingua-language-detector'], check=True)

from lingua import Language, LanguageDetectorBuilder

STEMS_USE = ['Gemini-2.5-Pro', 'Claude-4', 'DeepSeek-V3', 'GPT-4.1', 'Llama-4-Maverick', 'CommandA', 'Mistral-Medium']

# Build detector restricted to the 15 prompt languages only
# (restricting speeds up detection vs. the full 75-language model)
LINGUA_LANGS = [
    Language.CHINESE, Language.CZECH, Language.DUTCH, Language.ENGLISH,
    Language.ESTONIAN, Language.FRENCH, Language.GERMAN, Language.JAPANESE,
    Language.KOREAN, Language.PERSIAN, Language.PORTUGUESE, Language.RUSSIAN,
    Language.SPANISH, Language.SWEDISH, Language.UKRAINIAN,
]
detector = LanguageDetectorBuilder.from_languages(*LINGUA_LANGS).build()

# Map problem_language string → normalised name (matches Language.X.name.title())
PROB_LANG_NAMES = {lang.name.title() for lang in LINGUA_LANGS}
# Sanity check: all problem_language values in df should be in this set
unknown = set(df['problem_language'].unique()) - PROB_LANG_NAMES
assert not unknown, f"Unexpected problem_language values: {unknown}"

print("Detector built for:", sorted(l.name.title() for l in LINGUA_LANGS))
print("All problem_language values covered ✓")

In [ ]:
# ── Analysis 1 Step 1: Segment responses and detect reasoning language ─────────
_BRACKET_RE = re.compile(r'\[([^\[\]]*)\]')
MIN_CHARS = 10

# Unicode block ranges for the 15 prompt languages combined.
# Any alphabetic character OUTSIDE these ranges is IOL-exotic and its token gets stripped.
_ALLOWED_BLOCKS = (
    (0x0000, 0x024F),   # Basic Latin + Extended Latin A/B  (all Latin-script langs)
                        # NOTE: ŋ=U+014B, ə=U+0259, ņ=U+0146 are in U+0250+ — NOT included
    (0x0400, 0x04FF),   # Cyrillic  (Russian, Ukrainian)
    (0x0600, 0x06FF),   # Arabic    (Persian)
    (0x3000, 0x9FFF),   # CJK + Hiragana + Katakana  (Chinese, Japanese)
    (0xAC00, 0xD7FF),   # Hangul    (Korean)
)

def _is_allowed_char(ch: str) -> bool:
    cp = ord(ch)
    return any(lo <= cp <= hi for lo, hi in _ALLOWED_BLOCKS)

def strip_iol_tokens(text: str) -> str:
    """Remove tokens containing IOL-exotic characters (e.g. Koryak ŋ, ə, ņ)."""
    return ' '.join(t for t in text.split() if all(_is_allowed_char(c) for c in t if c.isalpha()))

def segment_response(text: str) -> str:
    """Return reasoning part = everything before the last [...]."""
    matches = list(_BRACKET_RE.finditer(text))
    if not matches:
        return text.strip()
    return text[:matches[-1].start()].strip()

def detect_lang(text: str):
    """Detect language after stripping IOL tokens. Returns title-cased name or None."""
    if not text or len(text) < MIN_CHARS:
        return None
    cleaned = strip_iol_tokens(text)
    if len(cleaned) < MIN_CHARS:
        return None
    lang = detector.detect_language_of(cleaned)
    return lang.name.title() if lang is not None else None

print("Running language identification across 6 models × 1,350 rows …")
for stem in STEMS_USE:
    lang_reasoning, lang_match = [], []

    for _, row in df.iterrows():
        raw       = row[f'raw_{stem}']
        prob_lang = row['problem_language']

        if not isinstance(raw, str) or not raw.strip():
            lang_reasoning.append(None)
            lang_match.append(None)
            continue

        lr = detect_lang(segment_response(raw))
        lm = int(lr == prob_lang) if lr is not None else None

        lang_reasoning.append(lr)
        lang_match.append(lm)

    df[f'lang_reasoning_{stem}'] = lang_reasoning
    df[f'lang_match_{stem}']     = lang_match   # 1=match, 0=mismatch, None=undetected

    n_detected = sum(r is not None for r in lang_reasoning)
    n_match    = sum(r == 1        for r in lang_match if r is not None)
    n_mismatch = sum(r == 0        for r in lang_match if r is not None)
    print(f"  {stem:25s}  detected: {n_detected:4d}/1350  match: {n_match:4d}  mismatch: {n_mismatch:4d}")

print(f"\ndf shape: {df.shape}")

# ── Materialise segment_response_{stem} column ──────────────────────────────
# Analysis 6 needs this as an explicit column (not just computed inline).
print('Materialising segment_response columns …')
for stem in STEMS_USE:
    df[f'segment_response_{stem}'] = df[f'raw_{stem}'].apply(
        lambda x: segment_response(x) if isinstance(x, str) and x.strip() else ''
    )
    n_nonempty = (df[f'segment_response_{stem}'].str.strip() != '').sum()
    print(f'  {stem}: {n_nonempty}/1350 non-empty segment_response')
print(f'df shape: {df.shape}')

In [ ]:
# ── Analysis 2 Step 1a: Load FLORES-200 devtest and compute char-length ratios ─
from pathlib import Path

FLORES_DIR = ROOT / 'data/flores200/flores200_dataset/devtest'

# Mapping: problem_language (as in df) → FLORES-200 language code
LANG_TO_FLORES = {
    'Chinese':    'zho_Hans',
    'Czech':      'ces_Latn',
    'Dutch':      'nld_Latn',
    'English':    'eng_Latn',
    'Estonian':   'est_Latn',
    'French':     'fra_Latn',
    'German':     'deu_Latn',
    'Japanese':   'jpn_Jpan',
    'Korean':     'kor_Hang',
    'Persian':    'pes_Arab',
    'Portuguese': 'por_Latn',
    'Russian':    'rus_Cyrl',
    'Spanish':    'spa_Latn',
    'Swedish':    'swe_Latn',
    'Ukrainian':  'ukr_Cyrl',
}

def load_flores_avg_len(lang_code: str) -> float:
    """Return mean character length of sentences in the FLORES-200 devtest file."""
    path = FLORES_DIR / f'{lang_code}.devtest'
    sentences = path.read_text(encoding='utf-8').splitlines()
    sentences = [s.strip() for s in sentences if s.strip()]
    return sum(len(s) for s in sentences) / len(sentences)

# Compute average lengths and ratios relative to English
avg_lens = {lang: load_flores_avg_len(code) for lang, code in LANG_TO_FLORES.items()}
eng_avg  = avg_lens['English']

flores_ratios = {lang: avg / eng_avg for lang, avg in avg_lens.items()}

# ── Print the ratios table ────────────────────────────────────────────────────
print(f'FLORES-200 devtest — average sentence length and ratio to English')
print(f'{"Language":<14} {"Avg chars":<12} {"Ratio vs English"}')
print('-' * 44)
for lang, avg in sorted(avg_lens.items(), key=lambda x: -x[1]):
    print(f'{lang:<14} {avg:<12.1f} {flores_ratios[lang]:.3f}')
print(f'\nEnglish baseline: {eng_avg:.1f} chars/sentence')


In [ ]:
# ── Analysis 2 Step 1b: Add char_len and norm_len columns to df ───────────────
# All 7 models included — char length is computed on raw_{stem} text directly,
# which contains the reasoning trace regardless of thinking tokens.
STEMS_LEN = STEMS

for stem in STEMS_LEN:
    df[f'char_len_{stem}'] = df[f'raw_{stem}'].apply(lambda x: len(str(x)))
    df[f'norm_len_{stem}']  = df.apply(
        lambda row: row[f'char_len_{stem}'] / flores_ratios[row['problem_language']],
        axis=1
    )

print(f'Added char_len / norm_len columns for {len(STEMS_LEN)} models.')
print(f'New df shape: {df.shape}')

# ── Sanity check: mean raw vs. normalised length per language (one model) ─────
SANITY_STEM = 'Claude-4'
print(f'\nSanity check — {TOP_MODELS[SANITY_STEM]}')
print(f'{"Language":<14} {"Ratio":<8} {"Mean raw len":<16} {"Mean norm len"}')
print('-' * 56)
for lang in sorted(LANG_TO_FLORES):
    sub = df[df['problem_language'] == lang]
    raw_mean  = sub[f'char_len_{SANITY_STEM}'].mean()
    norm_mean = sub[f'norm_len_{SANITY_STEM}'].mean()
    ratio     = flores_ratios[lang]
    print(f'{lang:<14} {ratio:<8.3f} {raw_mean:<16.0f} {norm_mean:.0f}')

print('\nVerification: norm_len = char_len / flores_ratio, so norm_len × ratio ≈ char_len')
print('(All norm_len values should be comparable across languages even when raw chars differ)')


---

---
## Analysis 5 — Intra-response Repetitiveness

**Question:** How much does each model repeat itself *within* a single response?

**Hypothesis:** Llama 4 Maverick writes ~2.7× longer responses than CommandA (2726 vs 933 normalised chars on average) but scores only marginally better (22.9 vs 19.8). A substantial part of Llama's extra length may be internal repetition / boilerplate rather than useful reasoning.

**Metrics:** Compression Ratio (CR) and Self-Repetition Score (SRS) from Shaib et al. 2024.

In [ ]:
# ── Cell 1: Imports for Analysis 5 ──────────────────────────────────────────
from diversity import compression_ratio, self_repetition_score
import nltk, re

nltk.download('punkt_tab', quiet=True)

STEMS_USE = ['Gemini-2.5-Pro', 'Claude-4', 'DeepSeek-V3', 'GPT-4.1',
             'Llama-4-Maverick', 'CommandA', 'Mistral-Medium']
print("Analysis 5 imports OK")

In [ ]:
# ── Cell 2: split_sentences definition + validation ──────────────────────────

CJK_LANGS = {"Japanese", "Chinese", "Korean"}

def split_sentences(text, language):
    if text is None or not str(text).strip() or str(text).strip() == 'nan':
        return []
    text = str(text)
    if language in CJK_LANGS:
        sentences = re.split(r'[.。!?！？\n]+', text)
    else:
        sentences = nltk.sent_tokenize(text)
    return [s.strip() for s in sentences if s.strip()]

# ── Validation ────────────────────────────────────────────────────────────────
test_languages = ["English", "French", "Chinese", "Japanese", "Korean", "Estonian", "Russian"]
for lang in test_languages:
    sample_rows = df[df["problem_language"] == lang].head(3)
    print(f"=== {lang} ===")
    for idx, row in sample_rows.iterrows():
        text = row["raw_Claude-4"]
        if not text or str(text).strip() in ('', 'nan'):
            continue
        sentences = split_sentences(text, lang)
        print(f"  Row {idx}: {len(str(text))} chars → {len(sentences)} sentences")
        for i, s in enumerate(sentences[:2]):
            print(f"    [{i}] {s[:80]}...")
    print()

In [ ]:
# ── Cell 3: Compression Ratio (CR) per item per model ───────────────────────

for stem in STEMS_USE:
    cr_vals = []
    for _, row in df.iterrows():
        raw = row[f'raw_{stem}']
        if raw is None or str(raw).strip() in ('', 'nan'):
            cr_vals.append(float('nan'))
            continue
        text = str(raw)
        if len(text) < 20:
            cr_vals.append(float('nan'))
            continue
        try:
            cr = compression_ratio([text], algorithm='gzip')
        except Exception:
            cr = float('nan')
        cr_vals.append(cr)
    df[f'compression_ratio_{stem}'] = cr_vals
    n_valid = sum(1 for v in cr_vals if v == v)
    print(f'  {TOP_MODELS[stem]:<22} CR computed for {n_valid}/1350 rows')

print(f'\ndf.shape: {df.shape}')  # should be (1350, 134)

In [ ]:
# ── Cell 4: Self-Repetition Score (SRS) + sentence count per item per model ──

for stem in STEMS_USE:
    srs_vals, nsent_vals = [], []
    for _, row in df.iterrows():
        raw  = row[f'raw_{stem}']
        lang = row['problem_language']
        if raw is None or str(raw).strip() in ('', 'nan'):
            srs_vals.append(float('nan'))
            nsent_vals.append(0)
            continue
        sentences = split_sentences(str(raw), lang)
        nsent_vals.append(len(sentences))
        if len(sentences) < 2:
            srs_vals.append(float('nan'))
        else:
            try:
                srs = self_repetition_score(sentences, verbose=False)
            except Exception:
                srs = float('nan')
            srs_vals.append(srs)
    df[f'self_repetition_{stem}'] = srs_vals
    df[f'n_sentences_{stem}']     = nsent_vals
    n_srs  = sum(1 for v in srs_vals  if v == v)
    print(f'  {TOP_MODELS[stem]:<22} SRS={n_srs}/1350  n_sent mean={sum(nsent_vals)/len(nsent_vals):.1f}')

print(f'\ndf.shape: {df.shape}')  # should be (1350, 148)

In [ ]:
# ── Cell 5: Descriptive stats per model ─────────────────────────────────────

import warnings
rows = []
for stem in STEMS_USE:
    cr  = df[f'compression_ratio_{stem}'].dropna()
    srs = df[f'self_repetition_{stem}'].dropna()
    ns  = df[f'n_sentences_{stem}']
    cl  = df[f'char_len_{stem}'] if f'char_len_{stem}' in df.columns else df[f'raw_{stem}'].apply(lambda x: len(str(x)))
    rows.append({
        'Model':         TOP_MODELS[stem],
        'n_items':       len(df),
        'CR_mean':       cr.mean(),
        'CR_median':     cr.median(),
        'SRS_mean':      srs.mean(),
        'SRS_median':    srs.median(),
        'n_sent_mean':   ns.mean(),
        'char_len_mean': cl.mean(),
        'SRS_NaN':       df[f'self_repetition_{stem}'].isna().sum(),
    })

stats_df = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print(stats_df.to_string(index=False))

In [ ]:
# ── Cell 7: Figure 1b — smoothed CR vs char_len, all models overlaid ─────────

XCLIP   = 6000
WINDOW  = 100

fig, ax = plt.subplots(figsize=(10, 5))

for i, stem in enumerate(STEMS_USE):
    sub = df[[f'char_len_{stem}', f'compression_ratio_{stem}']].dropna().copy()
    sub = sub[sub[f'char_len_{stem}'] <= XCLIP].sort_values(f'char_len_{stem}')
    if len(sub) < WINDOW:
        continue
    smoothed = sub[f'compression_ratio_{stem}'].rolling(WINDOW, min_periods=20).mean()
    ax.plot(sub[f'char_len_{stem}'], smoothed, label=TOP_MODELS[stem],
            color=f'C{i}', linewidth=2)

ax.set_xlabel('char_len (clipped at 6000)', fontsize=11)
ax.set_ylabel('Compression Ratio (rolling mean)', fontsize=11)
ax.set_title('Figure 1b — CR vs Response Length (smoothed, per model)', fontsize=12)
ax.legend(fontsize=9, loc='upper left')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis5_cr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved analysis5_cr_curves.png")

In [ ]:
# ── Figure 2 — CR distribution per model (responses > 200 chars) ─────────────
# Bucket approach dropped: fixed windows create severely imbalanced n per model
# (max/min ratio ~6.7x) because models differ structurally in response length.
# Instead: filter to responses with meaningful length (>200 chars excludes
# bare [answer]-only responses where CR is trivially ~1.0), then compare
# the full CR distribution per model via violin + strip plot.

MIN_CHARS = 200

violin_data = []
means       = {}
for stem in STEMS_USE:
    mask = df[f'char_len_{stem}'] > MIN_CHARS
    cr_vals = df.loc[mask, f'compression_ratio_{stem}'].dropna().values
    means[stem] = cr_vals.mean()
    for v in cr_vals:
        violin_data.append({'Model': TOP_MODELS[stem], 'CR': v, 'stem': stem})

vdf = pd.DataFrame(violin_data)

# Sort models by mean CR (highest first)
stems_sorted  = sorted(STEMS_USE, key=lambda s: means[s], reverse=True)
model_order   = [TOP_MODELS[s] for s in stems_sorted]

fig, ax = plt.subplots(figsize=(11, 5))

parts = ax.violinplot(
    [vdf[vdf['stem'] == s]['CR'].values for s in stems_sorted],
    positions=range(len(stems_sorted)),
    showmedians=True, showextrema=False, widths=0.7,
)
for pc in parts['bodies']:
    pc.set_alpha(0.55)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(1.5)

# Overlay mean dots
for i, stem in enumerate(stems_sorted):
    ax.scatter(i, means[stem], color='black', zorder=5, s=40, marker='D')

# Annotate n and mean
for i, stem in enumerate(stems_sorted):
    n = vdf[vdf['stem'] == stem].shape[0]
    ax.text(i, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 1.0,
            f'n={n}', ha='center', va='bottom', fontsize=7.5, color='#444')

ax.set_xticks(range(len(stems_sorted)))
ax.set_xticklabels(model_order, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Compression Ratio (gzip)', fontsize=11)
ax.set_title('CR distribution per model — responses > 200 chars\n(diamond = mean, line = median)',
             fontsize=11, fontweight='bold')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis5_cr_violin.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"Mean CR per model (responses > {MIN_CHARS} chars, sorted by CR):")
print(f"{'Model':<22} {'Mean CR':>8} {'Median CR':>10} {'n':>6}")
print("-" * 50)
for stem in stems_sorted:
    sub = vdf[vdf['stem'] == stem]['CR']
    print(f"{TOP_MODELS[stem]:<22} {sub.mean():>8.3f} {sub.median():>10.3f} {len(sub):>6}")


In [ ]:
# ── CR vs score: no correlation ─────────────────────────────────────────────
# Spearman correlation between compression ratio and score is inconsistent
# across models: only 2/7 show a significant negative relationship, and the
# direction flips for Llama (+0.20*) and CommandA (+0.07*).
# Finding: CR characterises how redundant a response is, but does NOT predict
# whether the model gets the answer right. Removed from the analysis.
from scipy import stats  # used by the SRS vs score cell below
print("CR vs score: no consistent correlation — CR excluded from predictive analysis.")

In [ ]:
# ── Cell 10 (revised): Figure 3a — Mean SRS per model, ordered by char_len ───

# Sort models by mean char_len (longest responses → leftmost)
char_len_means = {stem: df[f'char_len_{stem}'].mean() for stem in STEMS_USE}
stems_sorted   = sorted(STEMS_USE, key=lambda s: char_len_means[s], reverse=True)

model_labels, srs_means, char_means, srs_ses = [], [], [], []
for stem in stems_sorted:
    srs = df[f'self_repetition_{stem}'].dropna()
    srs_means.append(srs.mean())
    srs_ses.append(srs.std() / len(srs)**0.5)
    char_means.append(char_len_means[stem])
    model_labels.append(TOP_MODELS[stem])

colors = [f'C{STEMS_USE.index(s)}' for s in stems_sorted]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(len(stems_sorted)), srs_means, yerr=srs_ses, capsize=4,
              color=colors, alpha=0.85, error_kw={'elinewidth': 1.5})

for bar, cl, se in zip(bars, char_means, srs_ses):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + se + 0.006,
            f'{cl:.0f} chars', ha='center', va='bottom', fontsize=8.5, color='#333')

ax.set_xticks(range(len(stems_sorted)))
ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Mean Self-Repetition Score (±1 SE)', fontsize=11)
ax.set_title('Figure 3a — Mean SRS per model (ordered longest → shortest response)\n'
             '(label above bar = mean response length in characters)', fontsize=12)
ax.set_ylim(0, max(srs_means) * 1.35)
ax.yaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis5_srs_bar_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved analysis5_srs_bar_model.png")

In [ ]:
# ── Cell 11 (revised): Figure 3b — SRS across sentence-count buckets (slope chart) ─

def _sent_bucket_3b(n):
    if n < 5:   return 'short (<5)'
    if n <= 15: return 'medium (5-15)'
    return 'long (>15)'

bucket_keys   = ['short (<5)', 'medium (5-15)', 'long (>15)']
bucket_labels = ['Short\n(<5 sent)', 'Medium\n(5–15 sent)', 'Long\n(>15 sent)']
x_pos = [0, 1, 2]

slope_data = {}
for stem in STEMS_USE:
    sub = df[[f'n_sentences_{stem}', f'self_repetition_{stem}']].dropna().copy()
    sub['bucket'] = sub[f'n_sentences_{stem}'].apply(_sent_bucket_3b)
    means = []
    for bkt in bucket_keys:
        grp = sub[sub['bucket'] == bkt][f'self_repetition_{stem}']
        means.append(grp.mean() if len(grp) > 0 else np.nan)
    slope_data[stem] = means

fig, ax = plt.subplots(figsize=(9, 5))

for i, stem in enumerate(STEMS_USE):
    means = slope_data[stem]
    ax.plot(x_pos, means, marker='o', markersize=8, color=f'C{i}', lw=2.5)
    if not np.isnan(means[2]):
        ax.annotate(TOP_MODELS[stem], xy=(2, means[2]),
                    xytext=(2.08, means[2]), fontsize=9, color=f'C{i}', va='center')

ax.set_xticks(x_pos); ax.set_xticklabels(bucket_labels, fontsize=11)
ax.set_ylabel('Mean Self-Repetition Score', fontsize=11)
ax.set_xlim(-0.3, 3.2); ax.set_ylim(bottom=0)
ax.set_title('Figure 3b — SRS by sentence-count bucket (slope chart)\n(falling line = less phrase-recycling in longer responses)', fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis5_srs_slopes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved analysis5_srs_slopes.png")

In [ ]:
# ── Cell 12 (revised): Figure 4 — SRS by sentence bucket, 3 separate panels ──
# Models ordered left→right by average LR score (best → worst).

STEMS_BY_SCORE = [
    'Gemini-2.5-Pro',   # 36.3
    'Claude-4',          # 29.7
    'DeepSeek-V3',       # 23.6
    'GPT-4.1',           # 23.4
    'Llama-4-Maverick',  # 22.9
    'CommandA',          # 19.8
    'Mistral-Medium',    # lowest
]

def _sent_bucket_fig4(n):
    if n < 5:   return 'short'
    if n <= 15: return 'medium'
    return 'long'

bucket_defs = [
    ('short',  'Short responses\n(< 5 sentences)'),
    ('medium', 'Medium responses\n(5–15 sentences)'),
    ('long',   'Long responses\n(> 15 sentences)'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for ax_idx, (bkt_key, bkt_title) in enumerate(bucket_defs):
    ax = axes[ax_idx]
    vals, ses, ns = [], [], []

    for stem in STEMS_BY_SCORE:
        sub = df[[f'n_sentences_{stem}', f'self_repetition_{stem}']].dropna().copy()
        sub['bucket'] = sub[f'n_sentences_{stem}'].apply(_sent_bucket_fig4)
        grp = sub[sub['bucket'] == bkt_key][f'self_repetition_{stem}']
        n = len(grp)
        vals.append(grp.mean() if n > 0 else np.nan)
        ses.append(grp.std() / n**0.5 if n > 1 else 0)
        ns.append(n)

    colors = [f'C{STEMS_USE.index(s)}' for s in STEMS_BY_SCORE]
    bars = ax.bar(range(len(STEMS_BY_SCORE)), vals, yerr=ses, capsize=4,
                  color=colors, alpha=0.85, error_kw={'elinewidth': 1.5})

    for bar, v, se, n in zip(bars, vals, ses, ns):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + se + 0.005,
                    f'n={n}', ha='center', va='bottom', fontsize=7.5, color='#444')

    ax.set_xticks(range(len(STEMS_BY_SCORE)))
    ax.set_xticklabels([TOP_MODELS[s] for s in STEMS_BY_SCORE],
                       rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Mean SRS (±1 SE)' if ax_idx == 0 else '', fontsize=10)
    ax.set_title(bkt_title, fontsize=12)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

fig.suptitle('Figure 4 — Self-Repetition Score per model, by sentence-count bucket\n'
             '(models ordered by average LR score, best → worst)', fontsize=13)
plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis5_srs_buckets.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved analysis5_srs_buckets.png")

In [ ]:
# ── Cell 13 (revised): SRS vs Score — low-SRS vs high-SRS comparison ─────────
# For each model, split items at the per-model SRS median.
# Compare mean normalised score for low-SRS vs high-SRS responses.

low_scores, high_scores, labels = [], [], []

for stem in STEMS_USE:
    sub = df[[f'self_repetition_{stem}', f'score_{stem}', 'points']].dropna().copy()
    if len(sub) == 0:
        continue
    med = sub[f'self_repetition_{stem}'].median()
    lo  = sub[sub[f'self_repetition_{stem}'] <= med]
    hi  = sub[sub[f'self_repetition_{stem}'] >  med]
    low_scores.append(lo[f'score_{stem}'].sum() / lo['points'].sum() * 100 if len(lo) else np.nan)
    high_scores.append(hi[f'score_{stem}'].sum() / hi['points'].sum() * 100 if len(hi) else np.nan)
    labels.append(TOP_MODELS[stem])

x, bar_w = np.arange(len(labels)), 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - bar_w/2, low_scores,  bar_w, label='Low SRS (≤ median)', color='steelblue', alpha=0.85)
b2 = ax.bar(x + bar_w/2, high_scores, bar_w, label='High SRS (> median)', color='salmon',    alpha=0.85)

for bar, v in list(zip(b1, low_scores)) + list(zip(b2, high_scores)):
    if not np.isnan(v):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Mean normalised score (0–100)', fontsize=11)
ax.set_ylim(0, max(high_scores + low_scores) * 1.2)
ax.set_title('SRS vs Score — do responses with more phrase-recycling score lower?\n'
             '(split at per-model SRS median; lower bar = worse score)', fontsize=12)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis5_srs_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved analysis5_srs_vs_score.png")

## Analysis 5 — Summary

**Compression Ratio (CR):**
- **Llama 4 Maverick has the highest CR (2.52)** and GPT 4.1 is close behind (2.47); CommandA is the least redundant (2.01). Confirmed across all three length buckets — Llama's boilerplate persists at every length.
- CR vs score correlation is **not a clean negative signal**: only Gemini (ρ=−0.12*) and GPT 4.1 (ρ=−0.12*) show the expected negative relationship. Llama shows a *positive* CR–score correlation (ρ=+0.21*), suggesting that for Llama, longer/more redundant responses actually coincide with harder items where the model tries harder — the repetition may be exploratory rather than purely noise.

**Self-Repetition Score (SRS):**
- **GPT 4.1 has by far the highest SRS (0.47 mean)** — it recycles 4-gram phrases across sentences more than any other model. Llama (0.30) and CommandA (0.31) follow. Gemini has the lowest SRS (0.10).
- SRS vs score correlation is **consistently negative in 5/7 models** (all significant): Claude (ρ=−0.09*), DeepSeek (ρ=−0.07*), GPT 4.1 (ρ=−0.13*), Llama (ρ=−0.06*), Mistral (ρ=−0.07*). More phrase recycling within a response predicts lower scores. SRS is a more consistent signal than CR.

**Llama length paradox (Figure 5):**
- After dividing out CR to get `useful_len` (≈ non-redundant content), **Llama still has 2.4× more useful content than CommandA** (931 vs 389 chars). The ranking across all 7 models is unchanged. Repetition partially explains the gap — Llama's CR ratio (2.52) is higher than CommandA's (2.19) — but most of Llama's extra length is genuinely new content. The paradox stands: Llama writes significantly more non-redundant text than CommandA but scores only 3.4 points higher.

**Known limitations:**
- CJK sentence segmentation uses regex fallback — may under-segment long responses, inflating `n_sentences` and distorting SRS for Japanese/Chinese/Korean.
- NaN handling: 175 Gemini rows have no text (null responses), and 382 GPT-4.1 items are single-sentence (direct `[answer]` format) — excluded from SRS.
- CR grows mechanically with text length even for non-redundant text; bucket analysis controls for this but does not eliminate it.
- `useful_len = char_len / CR` is a rough proxy (gzip ratio ≠ semantic uniqueness); treat Figure 5 as directional, not precise.

---
## Analysis 6 — Between-response Diversity

**Research questions (from Julia):**
1. How uniform does a model approach a problem? Is its writing style consistent across all responses or highly varied?
2. Is it able to shift its style, words, structure from one task to another, or does it apply the same template everywhere?
3. How is it related to score: does more diversity predict a better score?

**Reference:** Shaib et al. 2024, *Standardizing the Measurement of Text Diversity* — same package as Analysis 5.

**Text field:** `segment_response_{stem}` (reasoning text, final bracket answer excluded).

**Three analyses:**
| Analysis | Set analysed | Question |
|---|---|---|
| **A. Global** | All ~1350 responses of one model | Who is globally most uniform? |
| **B. Per task type** | Responses for one model × task type | In which task type is each model most uniform? |
| **C. Cross-type** | Pairwise between task types | Between which types does each model write the same vs. differently? |

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
# ── Cell 1: Analysis 6 imports and helpers ──────────────────────────────────
from diversity import compression_ratio, self_repetition_score
import sacrebleu as _sacrebleu
import random
import numpy as np
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('punkt_tab', quiet=True)

random.seed(42)
np.random.seed(42)

SAMPLE_SIZE_BLEU = 150
TASK_TYPES_USE   = ['translation', 'mapping', 'fill-in-blanks', 'classification']
MIN_N_FOR_METRIC = 20   # skip metric if effective n < this

def safe_texts(series):
    """Filter None, empty, and whitespace-only strings."""
    return [t for t in series if t is not None and str(t).strip()]

def subsample(texts, n=SAMPLE_SIZE_BLEU, seed=42):
    """Reproducibly subsample texts; return unchanged if smaller than n."""
    if len(texts) <= n:
        return texts
    rng = random.Random(seed)
    return rng.sample(texts, n)

def text_to_pos_sequence(text):
    """Replace each token with its POS tag. Returns a space-joined tag string."""
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return ' '.join(tag for _, tag in tagged)

def cr_pos(texts):
    """Compression ratio on POS-tag sequences — captures syntactic template redundancy."""
    if not texts:
        return None
    pos_sequences = [text_to_pos_sequence(t) for t in texts if str(t).strip()]
    if not pos_sequences:
        return None
    return compression_ratio(pos_sequences, algorithm='gzip')

print('Analysis 6 imports and helpers OK')

In [ ]:
# ── Cell 2: Self-BLEU and Cross-BLEU wrappers ───────────────────────────────

def self_bleu(texts, sample_size=SAMPLE_SIZE_BLEU):
    """Mean BLEU(candidate=t, references=others) over all t in texts."""
    texts = subsample(texts, sample_size)
    if len(texts) < 2:
        return None
    # Truncate outlier-long texts to prevent pathological slowdowns
    texts = [t[:2000] for t in texts]
    scores = []
    for i, candidate in enumerate(texts):
        references = [texts[j] for j in range(len(texts)) if j != i]
        bleu = _sacrebleu.sentence_bleu(candidate, references)
        scores.append(bleu.score)
    return sum(scores) / len(scores)

def cross_bleu(texts_a, texts_b, sample_size=SAMPLE_SIZE_BLEU, symmetric=True):
    """
    Cross-BLEU(A, B): each candidate from A scored against references from B.
    If symmetric, return mean of Cross-BLEU(A, B) and Cross-BLEU(B, A).
    Higher = more similar writing between the two sets.
    """
    a = [t[:2000] for t in subsample(texts_a, sample_size)]
    b = [t[:2000] for t in subsample(texts_b, sample_size)]
    if not a or not b:
        return None
    def directional(cands, refs):
        scores = [_sacrebleu.sentence_bleu(c, refs).score for c in cands]
        return sum(scores) / len(scores)
    ab = directional(a, b)
    if not symmetric:
        return ab
    ba = directional(b, a)
    return (ab + ba) / 2

print('Self-BLEU and Cross-BLEU wrappers defined.')
print('Expected runtime for all BLEU metrics: 5–10 min.')

In [ ]:
# ── Cell 3: Analysis A — Global diversity per model ─────────────────────────
# higher value = less diverse across all four metrics

rows_global = []
for stem in STEMS_USE:
    col = f'segment_response_{stem}'
    texts = safe_texts(df[col])

    # CR and SRS: run on full set
    cr_val  = compression_ratio(texts, algorithm='gzip') if len(texts) >= MIN_N_FOR_METRIC else float('nan')
    srs_val = self_repetition_score(texts)            if len(texts) >= MIN_N_FOR_METRIC else float('nan')

    # Self-BLEU: subsampled
    sb_val  = self_bleu(texts) if len(texts) >= MIN_N_FOR_METRIC else None

    # CR:POS — English-prompt subset only (NLTK POS tagger is English-only)
    en_texts = safe_texts(df.loc[df['problem_language'] == 'English', col])
    crpos_val = cr_pos(en_texts) if len(en_texts) >= MIN_N_FOR_METRIC else float('nan')

    rows_global.append({
        'Model':      TOP_MODELS[stem],
        'n_texts':    len(texts),
        'n_english':  len(en_texts),
        'CR':         cr_val,
        'SRS':        srs_val,
        'Self_BLEU':  sb_val,
        'CR_POS_en':  crpos_val,
    })
    print(f'  {TOP_MODELS[stem]:<22} n={len(texts):4d}  CR={cr_val:.3f}  SRS={srs_val:.3f}  '
          f'SelfBLEU={(sb_val if sb_val is not None else float("nan")):.2f}  CR_POS={crpos_val:.3f}')

df_div_global = pd.DataFrame(rows_global)
print()
print('df_div_global (higher = less diverse):')
print(df_div_global.round(3).to_string(index=False))

In [ ]:
# ── Cell 4: Figure 1 — Global diversity bar chart (2×2) ─────────────────────

metrics = [
    ('CR',        'Compression Ratio\n(all languages)'),
    ('SRS',       'Self-Repetition Score\n(all languages)'),
    ('Self_BLEU', 'Self-BLEU\n(all languages, subsampled)'),
    ('CR_POS_en', 'CR on POS tags\n(English-prompt subset only)'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, (col, title) in zip(axes, metrics):
    vals   = df_div_global[col].values
    models = df_div_global['Model'].values
    colors = [f'C{i}' for i in range(len(models))]
    bars = ax.bar(range(len(models)), vals, color=colors, alpha=0.85)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=25, ha='right', fontsize=9)
    ax.set_title(f'{title}\n(higher = less diverse)', fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
    for bar, v in zip(bars, vals):
        if not (v != v):  # not NaN
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

fig.suptitle('Analysis 6A — Global Text Diversity per Model', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis6_global_diversity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis6_global_diversity.png')

In [ ]:
# ── Cell 5: Analysis B — Per-task-type diversity ────────────────────────────

rows_by_type = []
for stem in STEMS_USE:
    col = f'segment_response_{stem}'
    print(f'{TOP_MODELS[stem]}:')
    for ttype in TASK_TYPES_USE:
        mask  = df['type'] == ttype
        texts = safe_texts(df.loc[mask, col])
        n     = len(texts)
        if n < MIN_N_FOR_METRIC:
            print(f'  {ttype:<18} n={n} < {MIN_N_FOR_METRIC} → NaN')
            rows_by_type.append({'Model': TOP_MODELS[stem], 'task_type': ttype,
                                 'n_texts': n, 'CR': float('nan'),
                                 'SRS': float('nan'), 'Self_BLEU': float('nan')})
            continue
        cr_val  = compression_ratio(texts, algorithm='gzip')
        srs_val = self_repetition_score(texts)
        sb_val  = self_bleu(texts)
        rows_by_type.append({'Model': TOP_MODELS[stem], 'task_type': ttype,
                             'n_texts': n, 'CR': cr_val,
                             'SRS': srs_val, 'Self_BLEU': sb_val})
        print(f'  {ttype:<18} n={n:4d}  CR={cr_val:.3f}  SRS={srs_val:.3f}  SelfBLEU={sb_val:.2f}')

df_div_by_type = pd.DataFrame(rows_by_type)
print(f'\ndf_div_by_type shape: {df_div_by_type.shape}  (expected 28 rows)')

In [ ]:
# ── Cell 6: Figure 2 — Per-task-type diversity heatmaps ─────────────────────

# Sort models by global Self-BLEU (most uniform at top)
model_order = df_div_global.sort_values('Self_BLEU', ascending=False)['Model'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_b = [('CR', 'CR'), ('SRS', 'SRS'), ('Self_BLEU', 'Self-BLEU')]

for ax, (col, label) in zip(axes, metrics_b):
    pivot = df_div_by_type.pivot(index='Model', columns='task_type', values=col)
    pivot = pivot.reindex(model_order)   # most uniform at top
    pivot = pivot[TASK_TYPES_USE]        # fixed column order
    sns.heatmap(pivot.astype(float), annot=True, fmt='.2f',
                cmap='YlOrRd', ax=ax, linewidths=0.4,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{label}\n(higher = less diverse)', fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=25)

fig.suptitle('Analysis 6B — Diversity per Model × Task Type', fontsize=13)
plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis6_by_task_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis6_by_task_type.png')

In [ ]:
# ── Cell 7: Analysis C — Cross-type pairwise Cross-BLEU ─────────────────────
from itertools import combinations

rows_cross = []
for stem in STEMS_USE:
    col = f'segment_response_{stem}'
    print(f'{TOP_MODELS[stem]}:')
    subsets = {}
    for ttype in TASK_TYPES_USE:
        texts = safe_texts(df.loc[df['type'] == ttype, col])
        subsets[ttype] = texts

    # Diagonal: Self-BLEU for each type
    for ttype in TASK_TYPES_USE:
        texts = subsets[ttype]
        sb = self_bleu(texts) if len(texts) >= MIN_N_FOR_METRIC else float('nan')
        rows_cross.append({'Model': TOP_MODELS[stem], 'type_A': ttype,
                           'type_B': ttype, 'metric': 'self_bleu', 'value': sb})
        print(f'  Self-BLEU({ttype:<18}) = {sb:.3f}' if sb == sb else
              f'  Self-BLEU({ttype:<18}) = NaN')

    # Off-diagonal: Cross-BLEU for each pair
    for ta, tb in combinations(TASK_TYPES_USE, 2):
        a, b = subsets[ta], subsets[tb]
        cb = cross_bleu(a, b) if (len(a) >= MIN_N_FOR_METRIC and len(b) >= MIN_N_FOR_METRIC) else float('nan')
        rows_cross.append({'Model': TOP_MODELS[stem], 'type_A': ta,
                           'type_B': tb, 'metric': 'cross_bleu', 'value': cb})
        rows_cross.append({'Model': TOP_MODELS[stem], 'type_A': tb,
                           'type_B': ta, 'metric': 'cross_bleu', 'value': cb})  # symmetric
        print(f'  Cross-BLEU({ta} ↔ {tb}) = {cb:.3f}' if cb == cb else
              f'  Cross-BLEU({ta} ↔ {tb}) = NaN')

df_div_cross_type = pd.DataFrame(rows_cross)
print(f'\ndf_div_cross_type shape: {df_div_cross_type.shape}')

In [ ]:
# ── Cell 8: Figure 3 — Per-model 4×4 Cross-BLEU matrices ───────────────────

TYPE_LABELS = {'translation': 'Trans.', 'mapping': 'Map.',
               'fill-in-blanks': 'Fill.', 'classification': 'Class.'}
short_types = [TYPE_LABELS[t] for t in TASK_TYPES_USE]

# Determine shared colour scale
all_vals = df_div_cross_type['value'].dropna().values
vmin, vmax = all_vals.min(), all_vals.max()

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes_flat = axes.flatten()

for ax_i, stem in enumerate(STEMS_USE):
    ax = axes_flat[ax_i]
    sub = df_div_cross_type[df_div_cross_type['Model'] == TOP_MODELS[stem]]
    mat = pd.DataFrame(index=TASK_TYPES_USE, columns=TASK_TYPES_USE, dtype=float)
    for _, row in sub.iterrows():
        mat.loc[row['type_A'], row['type_B']] = row['value']
    mat = mat.astype(float)
    mat.index   = short_types
    mat.columns = short_types

    sns.heatmap(mat, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
                vmin=vmin, vmax=vmax, linewidths=0.5,
                cbar=False)
    ax.set_title(TOP_MODELS[stem], fontsize=10)
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

# Hide the last (unused) subplot
axes_flat[-1].set_visible(False)

fig.suptitle('Analysis 6C — Cross-BLEU matrices\n'
             'Diagonal = Self-BLEU (intra-type); Off-diagonal = Cross-BLEU (inter-type)\n'
             'Higher = more similar writing between those two task types', fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis6_cross_type_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis6_cross_type_matrices.png')

In [ ]:
# ── Cell 9: Figure 4 — Diversity vs score (global, n=7) ─────────────────────

# Mean LR score per model from df_agg_top
mean_scores = df_agg_top.loc['Average']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, div_col, div_label in [
    (axes[0], 'Self_BLEU', 'Self-BLEU (higher = less diverse)'),
    (axes[1], 'CR',        'Compression Ratio (higher = less diverse)'),
]:
    for i, row in df_div_global.iterrows():
        display_name = row['Model']
        score = mean_scores.get(display_name, float('nan'))
        div   = row[div_col]
        ax.scatter(div, score, s=80, color=f'C{i}', zorder=3)
        ax.annotate(display_name, (div, score),
                    textcoords='offset points', xytext=(6, 2), fontsize=8)
    ax.set_xlabel(div_label, fontsize=10)
    ax.set_ylabel('Mean LR score (from df_agg)', fontsize=10)
    ax.set_title(f'Diversity vs Score (n=7 — low statistical power)', fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis6_diversity_vs_score_global.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis6_diversity_vs_score_global.png')

In [ ]:
# ── Cell 10: Figure 5 — Diversity vs score (per task type, n=28) ────────────

# Compute mean score per (model, task_type) from df
score_by_type = []
for stem in STEMS_USE:
    for ttype in TASK_TYPES_USE:
        sub = df[df['type'] == ttype]
        mean_sc = sub[f'score_{stem}'].sum() / sub['points'].sum() * 100
        score_by_type.append({'Model': TOP_MODELS[stem], 'task_type': ttype,
                              'mean_score': mean_sc})
df_scores_type = pd.DataFrame(score_by_type)

merged = df_div_by_type.merge(df_scores_type, on=['Model', 'task_type'])

markers = {'translation': 'o', 'mapping': 's', 'fill-in-blanks': '^', 'classification': 'D'}
stem_to_color = {TOP_MODELS[s]: f'C{i}' for i, s in enumerate(STEMS_USE)}

fig, ax = plt.subplots(figsize=(11, 7))
for ttype in TASK_TYPES_USE:
    sub = merged[merged['task_type'] == ttype]
    for _, row in sub.iterrows():
        ax.scatter(row['Self_BLEU'], row['mean_score'],
                   marker=markers[ttype], color=stem_to_color[row['Model']],
                   s=90, zorder=3, linewidths=0.5, edgecolors='white')

# Legends
from matplotlib.lines import Line2D
model_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                         markersize=9, label=m)
                 for m, c in stem_to_color.items()]
type_handles  = [Line2D([0],[0], marker=mk, color='gray',
                         markersize=9, label=tt, linestyle='none')
                 for tt, mk in markers.items()]
leg1 = ax.legend(handles=model_handles, loc='upper left',  fontsize=8, title='Model')
ax.add_artist(leg1)
ax.legend(handles=type_handles,  loc='upper right', fontsize=8, title='Task type')

ax.set_xlabel('Self-BLEU (higher = less diverse writing in that task type)', fontsize=10)
ax.set_ylabel('Mean score (normalised 0–100)', fontsize=10)
ax.set_title('Analysis 6 — Diversity vs Score per (Model, Task Type)  n=28', fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis6_diversity_vs_score_by_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis6_diversity_vs_score_by_type.png')

## Analysis 6 — Summary

*Fill in after running the cells above.*

**Global uniformity (Analysis A):**
- <!-- which model is most/least uniform across the 4 metrics -->
- <!-- does CR:POS reveal patterns invisible to CR/SRS? -->

**Per task type (Analysis B):**
- <!-- which models show the strongest style shift between task types -->
- <!-- which models apply the same template everywhere -->

**Cross-type findings (Analysis C):**
- <!-- pick 2-3 most striking model × pair combinations -->

**Diversity vs score:**
- <!-- what the scatter plots show; be cautious (n=7 global, n=28 per-type) -->

**Limitations:**
- Language mixing in sets (composition identical across models, absolute values may be inflated for non-Latin scripts)
- Subsampling (150 texts) for BLEU metrics — reproducible via fixed seed=42
- n=7 for global scatter; n=28 for per-type scatter — low statistical power
- CR:POS computed only on English-prompt subset (~90 items per model); not directly comparable to CR/SRS/Self-BLEU on full 1350

In [ ]:
# ── Analysis 4 prerequisite — feature columns required by Analysis 7 ────────
# These columns (has_steps, has_self_correction, has_hedging, no_reasoning,
# n_quoted_iol) are computed in the main notebook's Analysis 4. We replicate
# the column-addition logic here so this notebook remains self-contained.
import warnings

STEMS_ALL = STEMS_USE

RE_STEPS     = re.compile(r'(?:step\s*[1-9]|##\s*step\s*[1-9]|^\s*[1-9][.)]\s|\b(?:first[,:]|second[,:]|third[,:]))', re.I | re.M)
RE_SELF_CORR = re.compile(r'(?:wait[,!\s]|actually[,!\s]|no[,!\s]|i made a mistake|let me reconsider|\bcorrection\b|i was wrong|let me re.?examine|on second thought)', re.I)
RE_HEDGING   = re.compile(r'(?:maybe|perhaps|possibly|i\u2019?m not sure|it seems|could be|i think\b)', re.I)

def _no_reasoning(text):
    s = text.strip()
    if not s: return True
    m = re.search(r'\[', s)
    return (m.start() < 20) if m else len(s) < 30

FEATURE_PATTERNS = {
    'has_steps':           lambda t: bool(RE_STEPS.search(t)),
    'has_self_correction': lambda t: bool(RE_SELF_CORR.search(t)),
    'has_hedging':         lambda t: bool(RE_HEDGING.search(t)),
    'no_reasoning':        _no_reasoning,
}

new_cols = {}
for stem in STEMS_ALL:
    raw = df[f'raw_{stem}'].fillna('').astype(str)
    for feat, fn in FEATURE_PATTERNS.items():
        new_cols[f'{feat}_{stem}'] = raw.apply(fn)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    df = pd.concat([df.drop(columns=[c for c in new_cols if c in df.columns], errors='ignore'),
                    pd.DataFrame(new_cols, index=df.index)], axis=1)
print('  Structural features added ✓')

# ── IOL token vocabulary + n_quoted_iol ──────────────────────────────────────
_STOPWORDS = {
    'the','and','for','are','its','has','not','but','with','this','that',
    'here','some','given','correct','english','word','into','from','them',
    'you','they','will','see','led','saw','can','may','was','any','their',
    'all','each','one','two','three','also','more','only','both','such',
}

def _extract_iol_tokens(prompt_text):
    tokens = set()
    ctx_m = re.search(r'Context:(.*?)(?=\nTask:|\Z)', prompt_text, re.DOTALL | re.I)
    ctx = ctx_m.group(1) if ctx_m else ''
    for tok in re.findall(r'[A-Za-z\u0080-\u02FF\u1E00-\u1EFF]+', ctx):
        if any(ord(c) > 127 for c in tok) and len(tok) >= 2:
            tokens.add(tok.lower())
    for m in re.finditer(r'^\s*\d+[.)]\s+([^\n]+)', ctx, re.M):
        iol_part = re.split(r'\s*[:→=]\s*', m.group(1))[0]
        for tok in re.findall(r"[A-Za-z'\u02bc\u02bb]+", iol_part):
            if len(tok) >= 3:
                tokens.add(tok.lower())
    return tokens - _STOPWORDS

iol_tokens_by_id = {}
for pid, grp in df_lr.groupby('id'):
    toks = set()
    for prompt in grp['prompt']:
        toks |= _extract_iol_tokens(prompt)
    iol_tokens_by_id[pid] = toks

_QUOTE_RE = re.compile(
    r'(?:["\u201c\u201e\u00ab\u300c`])(.{2,120}?)(?:["\u201d\u00bb\u300d`])'
    r"|'(.{2,60})'",
    re.DOTALL
)

def _count_quoted_iol(reasoning_text, puzzle_id):
    toks = iol_tokens_by_id.get(puzzle_id, set())
    if not toks: return 0
    quoted = ' '.join(
        (m.group(1) or m.group(2) or '').lower()
        for m in _QUOTE_RE.finditer(reasoning_text)
    )
    return sum(1 for t in toks if re.search(r'\b' + re.escape(t) + r'\b', quoted))

new_v2 = {}
for stem in STEMS_ALL:
    n_quoted = []
    for _, row in df.iterrows():
        reasoning = segment_response(str(row[f'raw_{stem}']))
        n_quoted.append(_count_quoted_iol(reasoning, row['id']))
    new_v2[f'n_quoted_iol_{stem}'] = n_quoted
    print(f'  {TOP_MODELS[stem]:<22} median quoted IOL tokens: {int(pd.Series(n_quoted).median())}')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    df = pd.concat([df, pd.DataFrame(new_v2, index=df.index)], axis=1)

print(f'Analysis 4 prerequisite complete — df shape: {df.shape}')

---
## Analysis 7 — Behavioral Adaptation Across Task Types

This analysis summarises each (model, task_type) cell by **9 behavioral features** — response length, reasoning structure, language use, and repetition — then measures how much each model **shifts** that behavioral signature across the 4 task types (translation, mapping, fill-in-blanks, classification).

Contrast with Analysis 6: Self-BLEU / Cross-BLEU captures surface n-gram overlap between responses. This analysis ignores vocabulary entirely and instead asks: *does the model write longer responses for translation than for classification? does it hedge more on mapping? does it self-correct more on fill-in-blanks?* A model that adapts its behavior to the task type should score higher — it signals metacognitive awareness of different task demands.

In [ ]:
# ── Cell B: dependency check + behavioral signature dataframe ────────────────
FEAT_SOURCE = {
    'norm_len':            'Analysis 2  (cell ~12 in this notebook)',
    'lang_match':          'Analysis 1  (cell ~10)',
    'has_steps':           'Analysis 4 prerequisite (cell 40)',
    'has_self_correction': 'Analysis 4 prerequisite (cell 40)',
    'has_hedging':         'Analysis 4 prerequisite (cell 40)',
    'no_reasoning':        'Analysis 4 prerequisite (cell 40)',
    'n_quoted_iol':        'Analysis 4 prerequisite (cell 40)',
    'compression_ratio':   'Analysis 5  (cell ~17 in this notebook — run CR cell)',
    'self_repetition':     'Analysis 5  (cell ~18 in this notebook — run SRS cell)',
}
REQUIRED_FEATS = list(FEAT_SOURCE.keys())
missing = []
for feat in REQUIRED_FEATS:
    for stem in STEMS_USE:
        col = f'{feat}_{stem}'
        if col not in df.columns:
            missing.append((col, FEAT_SOURCE[feat]))
            break   # one stem is enough to flag the feature
if missing:
    for col, src_cell in missing:
        print(f'  MISSING: {col}  →  run {src_cell}')
    raise ValueError(f'{len(missing)} feature(s) missing — see above for which cells to run first')
print('All required columns present ✓')

FEATURES_A7   = REQUIRED_FEATS
TASK_TYPES_A7 = ['translation', 'mapping', 'fill-in-blanks', 'classification']

rows_beh = []
for stem in STEMS_USE:
    for ttype in TASK_TYPES_A7:
        mask = df['type'] == ttype
        row = {'Model': TOP_MODELS[stem], 'task_type': ttype, 'n_items': mask.sum()}
        for feat in FEATURES_A7:
            row[feat] = df.loc[mask, f'{feat}_{stem}'].mean()
        row['mean_score'] = (df.loc[mask, f'score_{stem}'].sum()
                             / df.loc[mask, 'points'].sum() * 100)
        rows_beh.append(row)

df_behavior = pd.DataFrame(rows_beh)
print(f'df_behavior shape: {df_behavior.shape}, expected (28, 13)')
print(df_behavior.round(3).to_string(index=False))

In [ ]:
# ── Cell C: z-score features → df_behavior_z ────────────────────────────────
df_behavior_z = df_behavior.copy()
for feat in FEATURES_A7:
    mu  = df_behavior_z[feat].mean()
    sig = df_behavior_z[feat].std()
    df_behavior_z[feat] = (df_behavior_z[feat] - mu) / sig

means = df_behavior_z[FEATURES_A7].mean().round(3)
print('df_behavior_z feature columns mean ≈ 0:')
print(means)

In [ ]:
# ── Cell D: per-model adaptation score → df_adaptation ──────────────────────
from scipy.spatial.distance import euclidean

rows_adapt = []
for stem in STEMS_USE:
    sub  = df_behavior_z[df_behavior_z['Model'] == TOP_MODELS[stem]]
    vecs = sub[FEATURES_A7].values   # shape (4, 9)
    dists = [euclidean(vecs[i], vecs[j])
             for i, j in combinations(range(len(vecs)), 2)]
    adapt_score = float(np.mean(dists))
    mean_lr     = df_agg_top.loc['Average', TOP_MODELS[stem]]
    rows_adapt.append({'Model': TOP_MODELS[stem],
                       'adaptation_score': adapt_score,
                       'mean_lr_score': mean_lr})

df_adaptation = (pd.DataFrame(rows_adapt)
                   .sort_values('adaptation_score', ascending=False)
                   .reset_index(drop=True))

lo, hi = df_adaptation['adaptation_score'].min(), df_adaptation['adaptation_score'].max()
print(f'Adaptation scores range: [{lo:.2f}, {hi:.2f}]')
print(df_adaptation.round(3).to_string(index=False))

In [ ]:
# ── Cell E: Figure 1 — adaptation score horizontal bar chart ─────────────────
palette = sns.color_palette('colorblind', n_colors=len(STEMS_USE))
model_color = {TOP_MODELS[s]: palette[i] for i, s in enumerate(STEMS_USE)}

fig, ax = plt.subplots(figsize=(9, 5))
models_ord = df_adaptation['Model'].tolist()
scores_ord = df_adaptation['adaptation_score'].tolist()
colors_ord = [model_color[m] for m in models_ord]

bars = ax.barh(range(len(models_ord)), scores_ord, color=colors_ord, alpha=0.85)
for i, (bar, val) in enumerate(zip(bars, scores_ord)):
    ax.text(val + 0.01, i, f'{val:.2f}', va='center', fontsize=9)

ax.set_yticks(range(len(models_ord)))
ax.set_yticklabels(models_ord, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Mean pairwise Euclidean distance between z-scored behavioral vectors\n'
              '(higher = more adaptive)', fontsize=9)
ax.set_title('How much does each model shift behavior across task types?', fontsize=12)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis7_adaptation_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis7_adaptation_bar.png')

In [ ]:
# ── Cell F: Figure 2 — 7-panel behavioral heatmap grid ──────────────────────
COMPACT_LABELS = ['len', 'steps', 'self-corr', 'hedge',
                  'no-reas', 'quote', 'lang-match', 'CR', 'SRS']
adapt_lookup = dict(zip(df_adaptation['Model'], df_adaptation['adaptation_score']))

all_z_vals = df_behavior_z[FEATURES_A7].values
vmin, vmax = float(np.nanmin(all_z_vals)), float(np.nanmax(all_z_vals))
# Symmetrise for diverging colormap
vlim = max(abs(vmin), abs(vmax))
vmin, vmax = -vlim, vlim

fig, axes = plt.subplots(2, 4, figsize=(20, 9), constrained_layout=True)
axes_flat = axes.flatten()

for ax_i, stem in enumerate(STEMS_USE):
    ax  = axes_flat[ax_i]
    sub = df_behavior_z[df_behavior_z['Model'] == TOP_MODELS[stem]].copy()
    pivot = (sub.set_index('task_type')[FEATURES_A7]
               .reindex(TASK_TYPES_A7))
    pivot.columns = COMPACT_LABELS
    sns.heatmap(pivot.astype(float), ax=ax,
                cmap='RdBu_r', center=0, vmin=vmin, vmax=vmax,
                annot=True, fmt='.1f', annot_kws={'size': 7},
                cbar=False, linewidths=0.4)
    adapt_sc = adapt_lookup[TOP_MODELS[stem]]
    ax.set_title(f"{TOP_MODELS[stem]}\n(adaptation = {adapt_sc:.2f})", fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)
    ax.set_xlabel('')
    ax.set_ylabel('')

# Shared colorbar on the hidden 8th slot
axes_flat[-1].set_visible(False)
import matplotlib as mpl
sm = mpl.cm.ScalarMappable(cmap='RdBu_r',
                            norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
fig.colorbar(sm, ax=axes_flat[-1], fraction=0.6, pad=0.05, label='z-score')

fig.suptitle('Behavioral signature per (Model × Task Type)\n'
             'Z-scored across all 28 cells — red = above average for that feature, blue = below',
             fontsize=12)
plt.savefig(ROOT / 'figures/analysis7_behavioral_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis7_behavioral_heatmaps.png')

In [ ]:
# ── Cell G: Figure 3 — adaptation vs score scatter (n=7) ────────────────────
from scipy.stats import spearmanr

rho, pval = spearmanr(df_adaptation['adaptation_score'],
                      df_adaptation['mean_lr_score'])

fig, ax = plt.subplots(figsize=(7, 5))
for _, row in df_adaptation.iterrows():
    color = model_color[row['Model']]
    ax.scatter(row['adaptation_score'], row['mean_lr_score'],
               s=90, color=color, zorder=3)
    ax.annotate(row['Model'],
                (row['adaptation_score'], row['mean_lr_score']),
                textcoords='offset points', xytext=(6, 2), fontsize=8)

ax.set_xlabel('Adaptation score (mean pairwise behavioral distance)\n'
              'n=7 — directional signal only, not a significance test', fontsize=9)
ax.set_ylabel('Mean LR score (from df_agg)', fontsize=10)
ax.set_title(f'Adaptation vs LR score  '
             f'(Spearman \u03c1 = {rho:.2f}, p = {pval:.2f}, n=7)', fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'figures/analysis7_adaptation_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved analysis7_adaptation_vs_score.png')

## Analysis 7 — Summary

*Fill in after running the cells above.*

**Most/least adaptive models (Figure 1):**
- <!-- which model has the highest adaptation score -->
- <!-- which model is most uniform across task types -->

**Which features drive adaptation (Figure 2):**
- <!-- for each model, which feature rows show the strongest red/blue contrast -->
- <!-- e.g. does norm_len shift a lot between translation and classification? -->

**Adaptation vs score (Figure 3):**
- <!-- does higher adaptation_score predict higher mean_lr_score? -->
- <!-- note: n=7, directional only -->

**Limitations:**
- n=7 for global correlation — insufficient for significance testing
- `editing` task type excluded (only 15 items per model after splitting by metalanguage)
- Z-scoring is relative across all 28 (model, task_type) cells: a negative z-score means 'below average across all 7 models for that feature,' not 'low in absolute terms'
- `lang_match` has nulls for rows where lingua could not detect a language; `.mean()` skips them silently, so the effective n varies per cell